In [173]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh

from scipy.spatial import cKDTree, Delaunay
from scipy.interpolate import UnivariateSpline

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset, export_centerline


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

# offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")


in_folder = "segment_final/"    
out_folder = "spline_experiment/"

folder = "Alpha-spline_meshes/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)


## Load segments, pixel spacing etc

In [174]:

filename = "Segmentation_1.seg.nrrd"
export_name = base + folder + filename[:-8]

eyeball_id = 1
muslce_ids = [2,3,4,5]
pairs = [(0,2), (1,3)]

lense_id = 7

background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)


print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




(110, 150, 155)
pixel spacing =  [0.5 0.5 0.5]
id =  7
dimensions =  (110, 150, 155)
dimensions =  (110, 150, 155)
id =  7


## Create initial meshes

In [175]:
muscles = []

for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))


            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:

                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

                mask_eye = mask

            if id in muslce_ids:
                print("Muscle")

                muscles.append(mask)
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )

            if id == lense_id:
                mask = ndimage.median_filter(mask, size=3)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )


id =  1
65767
Problem edges: 0
Outer and inner surfaces written.
id =  2
8436
Muscle
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._surf_id_2.stl
id =  3
4587
Muscle
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._surf_id_3.stl
id =  4
7222
Muscle
Problem edges: 1
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._surf_id_4.stl
id =  5
6801
Muscle
Problem edges: 6
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._surf_id_5.stl
id =  6
6465
id =  7
1372
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._surf_id_

## Lense axis

In [176]:
mesh = trimesh.load(export_name +"_surf_id_"+str(lense_id)+".stl")
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


def axis(t):
    return lense_center + t * axis_direction


axis_points = [axis(-20.0), axis(20.0)]

export_centerline(axis_points, 1.0 , base + folder + "lense_axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point =  axis(-2.25)/pixel_spacing[0]

print("target_point = ", target_point)

lense_center =  [34.8451185  63.80259441 24.46803607]
target_point =  [ 69.49465974 123.21527704  49.90582674]


## Section curve

- eyeball mesh ∩ place perpendicular to lense axis
- output: section curve

In [177]:
eyeball_mesh = trimesh.load(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")

section = eyeball_mesh.section(
    plane_origin=target_point*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_A.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

## Section curve B & radial expansion


In [178]:
target_point_B =  axis(-10)/pixel_spacing[0]

section_B = eyeball_mesh.section(
    plane_origin=target_point_B*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve_B = []

for entity in section_B.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve_B.append([idx[i], idx[i+1]])


# Center point in physical space 
center_physical = target_point_B * pixel_spacing[0]

original_vertices = numpy.array(section_B.vertices)

# Radial vectors from the center 
radial_vectors = original_vertices - center_physical

magnitudes = numpy.linalg.norm(radial_vectors, axis=1, keepdims=True)
normalized_vectors = radial_vectors / (magnitudes + 1e-8)



# Expand outward by xx mm 
expansion_distance = 2.1  
expanded_vertices = original_vertices + (normalized_vectors * expansion_distance)

section_B.vertices = expanded_vertices

# 6. Rebuild the section_curve line segments using the new expanded coordinates
expanded_section_curve = []
for entity in section_B.entities:
    idx = entity.points  # ordered indices within this specific entity
    
    for i in range(len(idx) - 1):
        # We map the local entity indices directly to our expanded_vertices array
        expanded_section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_B.vtk",
    meshio.Mesh(
        points=section_B.vertices,
        cells=[("line", numpy.array(expanded_section_curve))]
    )
)

## Functions: 
### Fit centerlines with/without additional constraints

In [179]:
def fit_target_constrained_spline(binned_centers, target_point, smoothing_factor=1.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    
def fit_3_constrained_spline(binned_centers, target_1, target_2, target_3, smoothing_factor=1.5):
    """
    Fits a spline through binned centers and 3 target guiding points,
    """
    # 1. Combine all points into one unordered pool
    target_1 = numpy.array(target_1).reshape(1, 3)
    target_2 = numpy.array(target_2).reshape(1, 3)
    target_3 = numpy.array(target_3).reshape(1, 3)
    
    unordered_points = numpy.concatenate([binned_centers, target_1], axis=0)
    
    # 2. Use PCA  to find the true main orientation axis
    pca = PCA(n_components=1)
    projection = pca.fit_transform(unordered_points)
    
    # 3. Sort ALL points based on their position along this principal axis
    sort_idx = numpy.argsort(projection.flatten())
    constrained_points = unordered_points[sort_idx]

    constrained_points = numpy.concatenate([constrained_points, target_2, target_3], axis=0)
    
    # 4. Flip check (Ensure Target 3 is at the end)
    distances_to_end = numpy.linalg.norm(constrained_points - target_3, axis=1)
    if numpy.argmin(distances_to_end) == 0:
        constrained_points = numpy.flip(constrained_points, axis=0)

    # 5. Assign Weights dynamically by finding where the targets landed
    weights = numpy.ones(len(constrained_points))
    
    for target in [target_1]:
        distances = numpy.linalg.norm(constrained_points - target, axis=1)
        matched_idx = numpy.argmin(distances)
        weights[matched_idx] = 50
    
    weights[0] = 20
    weights[-1] = 20

    # 6. Fit the Spline
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # Evaluate slightly past the boundaries to allow mathematical extension
        u_new = numpy.linspace(-0.08, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    
    except Exception as e:
        print(f"PCA-Ordered 3-point constrained spline fit failed: {e}")
        return None
    

def fit_dual_constrained_spline(binned_centers, target_1, target_2, smoothing_factor=1.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_1: (3,) array where the object should begin
    target_2: (3,) array where the object should end
    """
    # 1. Prepare the sequence: [Start, Binned Data, End]
    points = [
        binned_centers,
        numpy.array(target_1).reshape(1, 3),
        numpy.array(target_2).reshape(1, 3)
    ]
    constrained_points = numpy.concatenate(points, axis=0)
    
    # 2. Assign Weights
    # Weights[0] is Start, Weights[-1] is End. 
    # High values force the spline to hit those exact coordinates.
    weights = numpy.ones(len(constrained_points))
    weights[0] = 50.0  # Force start
    weights[-1] = 50.0 # Force end
    
    # 3. Fit the Spline
    # We pass the weights array to 'w'
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # 4. Evaluate
        # u=0.0 will be target_start, u=1.0 will be target_end
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Dual constrained spline fit failed: {e}")
        return None

def fit_analytical_centerline(mask, target_point, ignored_nodes, n_bins=20, extension_factor=0.2):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component 
        projection = pca.fit_transform(points)
        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        # # Smoothing 's' can be lower here because the binning already cleaned the noise
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        # # Evaluating from 0.0 to 1+extension extends the centerline
        smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0, 100), tck)).T

        if len(target_point)==0:
            return smooth_centerline
        
        if len(target_point)==1:
            smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point[0])
            return smooth_centerline, smooth_centerline_extended
        
        if len(target_point)==2:
            smooth_centerline_extended = fit_dual_constrained_spline(binned_centers, target_point[0], target_point[1] )
            return smooth_centerline, smooth_centerline_extended

        if len(target_point)==3:
            smooth_centerline_extended = fit_3_constrained_spline(binned_centers, target_point[0], target_point[1], target_point[2] )
            return smooth_centerline, smooth_centerline_extended    


## Functions:
### Get insertion points ~ section curve / fitted planes

In [180]:
def fit_plane_to_two_centerlines(centerline1, centerline2):
    # 1. Combine all points from both centerlines into one cloud
    points = numpy.vstack((centerline1, centerline2))
    
    # 2. Calculate the centroid (the average point)
    centroid = numpy.mean(points, axis=0)
    
    # 3. Center the points by subtracting the centroid
    centered_points = points - centroid
    
    # 4. Use Singular Value Decomposition
    _, _, vh = numpy.linalg.svd(centered_points)
    
    # 5. The normal vector 'n' is the last row of Vh 
    # (the direction with the least variance/spread)
    normal = vh[-1]
    
    return centroid, normal/numpy.linalg.norm(normal)

def find_plane_curve_intersections(section, section_curve, plane_centroid, plane_normal):
    intersections = []
    
    # Normalize plane normal just in case
    plane_normal = plane_normal / numpy.linalg.norm(plane_normal)
    
    vertices = section.vertices
    
    for start_idx, end_idx in section_curve:
        p1 = vertices[start_idx]
        p2 = vertices[end_idx]
        
        # Signed distances to the plane
        d1 = numpy.dot(p1 - plane_centroid, plane_normal)
        d2 = numpy.dot(p2 - plane_centroid, plane_normal)
        
        # Check if the segment crosses the plane
        if d1 * d2 < 0:
            # Linear interpolation to find the exact crossing point
            t = abs(d1) / (abs(d1) + abs(d2))
            intersect_pt = p1 + t * (p2 - p1)
            intersections.append(intersect_pt)
            
    return numpy.array(intersections)



In [181]:
extended_centerlines = []
centerlines = []
extended_centerlines_1 = []

# target_point = point on the lense axis

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    ignored_nodes = 0.01

    centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, n_bins=10, extension_factor=0.15)
    
    centerlines.append(centerline)
    extended_centerlines.append(centerline_ext_1)


    export_centerline(centerline_ext_1, pixel_spacing[0], base + folder + "centerline_exttended_1_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing[0], base + folder + "centerline_init_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])


point_hor, normal_vector_hor = fit_plane_to_two_centerlines(extended_centerlines[pairs[0][0]], extended_centerlines[pairs[0][1]])
point_vert, normal_vector_vert = fit_plane_to_two_centerlines(extended_centerlines[pairs[1][0]], extended_centerlines[pairs[1][1]])

print("Check of angle  = ", numpy.dot(normal_vector_hor, normal_vector_vert))

# Plane 1 intersections
pts_plane1 = find_plane_curve_intersections(section, section_curve, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2 intersections
pts_plane2 = find_plane_curve_intersections(section, section_curve, point_vert*pixel_spacing[0], normal_vector_vert)


# Plane 1B intersections
pts_plane1_B = find_plane_curve_intersections(section_B, section_curve_B, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2B intersections
pts_plane2_B = find_plane_curve_intersections(section_B, section_curve_B, point_vert*pixel_spacing[0], normal_vector_vert)

for muscle_id in range(len(muslce_ids)):

    if muscle_id == pairs[0][0]:
        target_1 = pts_plane1[1]/pixel_spacing[0]
        target_1_B = pts_plane1_B[1]/pixel_spacing[0]

    elif muscle_id == pairs[0][1]:
        target_1 = pts_plane1[0]/pixel_spacing[0]
        target_1_B = pts_plane1_B[0]/pixel_spacing[0]

    if muscle_id == pairs[1][1]:
        target_1 = pts_plane2[0]/pixel_spacing[0]
        target_1_B = pts_plane2_B[0]/pixel_spacing[0]

    elif muscle_id == pairs[1][0]:
        target_1 = pts_plane2[1]/pixel_spacing[0]
        target_1_B = pts_plane2_B[1]/pixel_spacing[0]

    mask = muscles[muscle_id]

    _, centerline_ext_2 = fit_analytical_centerline(mask, [target_1_B, target_1, target_point], ignored_nodes, n_bins=10, extension_factor=0.25)

    extended_centerlines_1.append(centerline_ext_2)


    export_centerline(centerline_ext_2, pixel_spacing[0], base + folder + "centerline_exttended_2_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])



Check of angle  =  0.051395253181876606


## Find intersection of the extended centerline and the eyeball


In [182]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


# Part II

In [183]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.9 else numpy.array([0, 1, 0])
    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2

In [184]:
# =========================================================================
# 1. CORE PROCESSING FUNCTIONS (Path-Length Alpha Shape Parameterization)
# =========================================================================

def extract_alpha_path_profiles_as_radii(centerline, init_mesh, pixel_spacing, slab_thickness=5, n_theta=24, alpha=0.15):
    """
    Computes alpha shapes along the centerline, but maps them back into a clean,
    monotonic radial array 'r' matching a 0 to 2*pi theta grid.
    
    This matches the exact data shape your original Fourier pipeline expected!
    """
    mesh_points = numpy.array(init_mesh.vertices / pixel_spacing[0])
    tree = cKDTree(mesh_points)
    
    radii_all = []
    
    # This must match the exact grid your original mesh generator uses!
    theta_uniform = numpy.linspace(0, 2 * numpy.pi, n_theta, endpoint=False)

    for i, p in enumerate(centerline):
        direction = (
            centerline[min(i + 1, len(centerline) - 1)]
            - centerline[max(i - 1, 0)]
        )
        direction /= (numpy.linalg.norm(direction) + 1e-8)
        v1, v2 = get_local_frame(direction)

        idx = tree.query_ball_point(p, r=45)
        local_pts = mesh_points[idx] - p
        dist_plane = numpy.dot(local_pts, direction)
        
        # Using your tuned slab thickness
        slab = numpy.abs(dist_plane) < slab_thickness
        local_pts = local_pts[slab]

        if len(local_pts) < 10:
            radii_all.append(None)
            continue

        x = local_pts @ v1
        y = local_pts @ v2
        pts_2d = numpy.column_stack((x, y))

        # Compute Alpha Shape via Scipy Delaunay
        tri = Delaunay(pts_2d)
        edges = set()
        for ia, ib, ic in tri.simplices:
            pa, pb, pc = pts_2d[ia], pts_2d[ib], pts_2d[ic]
            a = numpy.linalg.norm(pa - pb)
            b = numpy.linalg.norm(pb - pc)
            c = numpy.linalg.norm(pc - pa)
            s = (a + b + c) / 2.0
            area = numpy.sqrt(max(s * (s - a) * (s - b) * (s - c), 1e-10))
            
            if (a * b * c) / (4.0 * area) < (1.0 / alpha):
                for edge in [(ia, ib), (ib, ic), (ic, ia)]:
                    sorted_edge = tuple(sorted(edge))
                    if sorted_edge in edges: edges.remove(sorted_edge)
                    else: edges.add(sorted_edge)

        edge_dict = {}
        for ui, uj in edges:
            edge_dict.setdefault(ui, []).append(uj)
            edge_dict.setdefault(uj, []).append(ui)

        if not edge_dict:
            radii_all.append(None)
            continue

        start_node = next(iter(edge_dict.keys()))
        path = [start_node]
        curr_node, prev_node = start_node, None
        while True:
            neighbors = edge_dict[curr_node]
            next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1] if len(neighbors) > 1 else None
            if next_node is None or next_node == start_node: break
            path.append(next_node)
            prev_node, curr_node = curr_node, next_node
            if len(path) > len(edge_dict): break

        path.append(start_node)
        hull_pts = pts_2d[path]


            
        # SANITY CHECK 2: Does the contour actually enclose the centerline?
        # Calculate the bounding box of the traced perimeter
        min_x, max_x = numpy.min(hull_pts[:, 0]), numpy.max(hull_pts[:, 0])
        min_y, max_y = numpy.min(hull_pts[:, 1]), numpy.max(hull_pts[:, 1])
        
        # If the centerline origin (0,0) sits completely outside the shape's bounds,
        # the angle tracking will break. Mark as None!
        if not (min_x < 0 < max_x and min_y < 0 < max_y):
            radii_all.append(None)
            continue


        # ---------------------------------------------------------------------
        # Convert the alpha path points to angles and radii relative to the center
        # ---------------------------------------------------------------------
        hull_thetas = numpy.arctan2(hull_pts[:, 1], hull_pts[:, 0])
        hull_radii = numpy.sqrt(hull_pts[:, 0]**2 + hull_pts[:, 1]**2)

        # Un-wrap angles from [-pi, pi] to [0, 2*pi] to match your mesh generator
        hull_thetas = (hull_thetas + 2 * numpy.pi) % (2 * numpy.pi)

        # Sort strictly by angle to create a clean, monotonic mapping
        sort_idx = numpy.argsort(hull_thetas)
        sorted_thetas = hull_thetas[sort_idx]
        sorted_radii = hull_radii[sort_idx]

        # Interpolate the radius values onto your uniform 0 to 2*pi grid safely
        r_parametric = numpy.interp(
            theta_uniform, 
            sorted_thetas, 
            sorted_radii, 
            period=2*numpy.pi
        )

        radii_all.append(r_parametric)

    return radii_all


def extract_alpha_path_profiles(centerline, init_mesh, pixel_spacing, n_theta=24, alpha=0.15):
    """
    Extracts geometric boundary profiles along the centerline using Alpha Shapes,
    parameterized by sequential cumulative path length.
    Returns an array of shape (n_slices, n_theta, 2) containing local (x, y) coordinates.
    """
    mesh_points = numpy.array(init_mesh.vertices / pixel_spacing[0])
    tree = cKDTree(mesh_points)
    
    profiles_all = []
    
    for i, p in enumerate(centerline):
        # Calculate local backbone direction
        direction = (
            centerline[min(i + 1, len(centerline) - 1)]
            - centerline[max(i - 1, 0)]
        )
        direction /= (numpy.linalg.norm(direction) + 1e-8)
        v1, v2 = get_local_frame(direction)

        # Slice extracting logic using your optimized slab thickness (< 2)
        idx = tree.query_ball_point(p, r=45)
        local_pts = mesh_points[idx] - p
        dist_plane = numpy.dot(local_pts, direction)
        
        # TUNING FIX: Reduced slab thickness to capture thin localized profiles
        slab = numpy.abs(dist_plane) < 1.5
        local_pts = local_pts[slab]

        if len(local_pts) < 15:  # Slightly lowered requirement for thinner slices
            profiles_all.append(None)
            continue

        x = local_pts @ v1
        y = local_pts @ v2
        pts_2d = numpy.column_stack((x, y))

        # Compute Alpha Shape via Scipy Delaunay Triangulation
        tri = Delaunay(pts_2d)
        edges = set()
        for ia, ib, ic in tri.simplices:
            pa, pb, pc = pts_2d[ia], pts_2d[ib], pts_2d[ic]
            a = numpy.linalg.norm(pa - pb)
            b = numpy.linalg.norm(pb - pc)
            c = numpy.linalg.norm(pc - pa)
            s = (a + b + c) / 2.0
            area = numpy.sqrt(max(s * (s - a) * (s - b) * (s - c), 1e-10))
            
            if (a * b * c) / (4.0 * area) < (1.0 / alpha):
                for edge in [(ia, ib), (ib, ic), (ic, ia)]:
                    sorted_edge = tuple(sorted(edge))
                    if sorted_edge in edges:
                        edges.remove(sorted_edge)
                    else:
                        edges.add(sorted_edge)

        # Trace perimeter unique edges into a sequential node loop
        edge_dict = {}
        for ui, uj in edges:
            edge_dict.setdefault(ui, []).append(uj)
            edge_dict.setdefault(uj, []).append(ui)

        if not edge_dict:
            profiles_all.append(None)
            continue

        start_node = next(iter(edge_dict.keys()))
        path = [start_node]

        curr_node, prev_node = start_node, None
        while True:
            neighbors = edge_dict[curr_node]
            next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1] if len(neighbors) > 1 else None
            if next_node is None or next_node == start_node: 
                break
            path.append(next_node)
            prev_node, curr_node = curr_node, next_node
            if len(path) > len(edge_dict): 
                break

        path.append(start_node)
        hull_pts = pts_2d[path]
        

        # Parameterize by path length (safely samples non-convex profiles uniformly)
        segment_lengths = numpy.sqrt(numpy.sum(numpy.diff(hull_pts, axis=0)**2, axis=1))
        cumulative_distance = numpy.concatenate(([0], numpy.cumsum(segment_lengths)))
        total_length = cumulative_distance[-1]
        
        # TUNING FIX: Evaluated using your highly efficient target node count
        target_distances = numpy.linspace(0, total_length, n_theta, endpoint=False)
        
        parametric_x = numpy.interp(target_distances, cumulative_distance, hull_pts[:, 0])
        parametric_y = numpy.interp(target_distances, cumulative_distance, hull_pts[:, 1])

        profiles_all.append(numpy.column_stack((parametric_x, parametric_y)))

    return profiles_all


def smooth_parametric_profiles(profiles_all, n_output, smoothing=0.2):
    """
    Flattens the coordinate descriptors and runs Univariate Splines to smoothly
    interpolate and extend shapes along the output centerline length.
    """
    n_input = len(profiles_all)
    
    # CRITICAL FIX: Find the first non-None slice to determine n_theta safely
    valid_slice = next((p for p in profiles_all if p is not None), None)
    
    if valid_slice is None:
        print("Warning: All slices in this muscle returned None! Cannot compute mesh.")
        return None
        
    n_theta = valid_slice.shape[0]
    
    # Flatten profiles from shape (n_theta, 2) to a 1D vector of length (2 * n_theta)
    flat_features = numpy.array([p.flatten() if p is not None else numpy.full(2 * n_theta, numpy.nan) for p in profiles_all])
    
    # 1. Fill internal NaNs using linear interpolation across slices
    for j in range(flat_features.shape[1]):
        valid = ~numpy.isnan(flat_features[:, j])
        if numpy.any(valid):
            # Using numpy.interp to fill internal and edge NaNs safely
            flat_features[:, j] = numpy.interp(
                numpy.arange(n_input), 
                numpy.where(valid)[0], 
                flat_features[valid, j]
            )
        else:
            flat_features[:, j] = 0.0  # Fallback if entire column is empty

    # 2. Setup relative spline coordinates
    s_in = numpy.linspace(0, 1, n_input)
    s_out = numpy.linspace(0, 1, n_output)

    smoothed_flat = numpy.zeros((n_output, 2 * n_theta))
    
    # 3. Fit 1D Univariate spline to each coordinate feature across space
    for j in range(2 * n_theta):
        spl = UnivariateSpline(s_in, flat_features[:, j], k=3, s=smoothing)
        smoothed_flat[:, j] = spl(s_out)
        
    # Unflatten back to a clean (n_output, n_theta, 2) positional array
    return smoothed_flat.reshape((n_output, n_theta, 2))


def smooth_alpha_radii(radii_all, n_output, smoothing=0.2):
    """
    Cleans, interpolates empty sections, and fits 1D Univariate Splines to 
    each angular radius bin across the muscle centerline length.
    """
    n_input = len(radii_all)
    
    # 1. Find the first valid slice to figure out the target n_theta configuration
    valid_slice = next((r for r in radii_all if r is not None), None)
    if valid_slice is None:
        print("Warning: All slices in this muscle returned None! Cannot compute mesh.")
        return None
    n_theta = len(valid_slice)
    
    # 2. Convert raw list to a structured NumPy array, turning None slices into NaNs
    radii_matrix = numpy.array([r if r is not None else numpy.full(n_theta, numpy.nan) for r in radii_all])
    
    # 3. Fill internal empty slices (NaNs) cleanly across spatial slices
    for j in range(n_theta):
        valid_mask = ~numpy.isnan(radii_matrix[:, j])
        if numpy.any(valid_mask):
            radii_matrix[:, j] = numpy.interp(
                numpy.arange(n_input), 
                numpy.where(valid_mask)[0], 
                radii_matrix[valid_mask, j]
            )
        else:
            radii_matrix[:, j] = 0.1  # Absolute fallback security layer
            
    # 4. Set up spatial relative nodes for input data and extended output tracking
    s_in = numpy.linspace(0, 1, n_input)
    s_out = numpy.linspace(0, 1, n_output)

    smoothed_radii = numpy.zeros((n_output, n_theta))
    
    # 5. Fit a smooth 1D cubic spline to every individual radial angle bin across space
    for j in range(n_theta):
        spline = UnivariateSpline(s_in, radii_matrix[:, j], k=3, s=smoothing)
        smoothed_radii[:, j] = spline(s_out)
        
    return smoothed_radii


In [185]:
def generate_alpha_tube_mesh(
        centerline,
        radii_smooth,  # Passed directly from your smoothing function
        pixel_spacing,
        n_theta=24,
        insertion_idx=None,
        taper_start_len=12,
        taper_end_len=12):
    """
    Generates a continuous 3D tube mesh from sequential alpha-shape radii.
    Maintains original topology logic for smooth end tapers.
    """
    n_sections = len(centerline)

    if insertion_idx is None:
        insertion_idx = n_sections

    n_sections = min(n_sections, insertion_idx)

    vertices = []

    # Target angle grid matching the extraction step
    theta = numpy.linspace(0, 2 * numpy.pi, n_theta, endpoint=False)

    # Endpoints for both collapses
    start_point = centerline[0]
    end_point = centerline[n_sections - 1]

    for i in range(n_sections):
        # ---------------------------------
        # Local backbone frame
        # ---------------------------------
        direction = (
            centerline[min(i+1, n_sections-1)]
            - centerline[max(i-1, 0)]
        )
        direction /= (numpy.linalg.norm(direction) + 1e-8)

        v1, v2 = get_local_frame(direction)

        # CRITICAL FIX: Read the pre-computed alpha radius row directly!
        r = radii_smooth[i]
        r = numpy.maximum(r, 0.05)  # Your original safety threshold

        P = centerline[i]

        # =========================================================
        # PROXIMAL TAPER (START)
        # =========================================================
        if i < taper_start_len:
            t = i / max(taper_start_len - 1, 1)   # 0 → 1
            scale = t**2

            P = (1 - t) * start_point + t * P
            r = r * scale

        # =========================================================
        # DISTAL TAPER (END)
        # =========================================================
        if i >= n_sections - taper_end_len:
            t = (i - (n_sections - taper_end_len)) / max(taper_end_len - 1, 1)
            scale = (1 - t)**2

            P = (1 - t) * P + t * end_point
            r = r * scale

        # ---------------------------------
        # Ring Construction (Identical to your original formula)
        # ---------------------------------
        ring = (
            P[None, :]
            + r[:, None] * numpy.cos(theta)[:, None] * v1[None, :]
            + r[:, None] * numpy.sin(theta)[:, None] * v2[None, :]
        )

        vertices.append(ring)

    vertices = numpy.vstack(vertices)

    # ---------------------------------
    # Connectivity (Stitching the Tube Walls)
    # ---------------------------------
    faces = []

    for i in range(n_sections - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta

        for j in range(n_theta):
            j2 = (j + 1) % n_theta

            v00 = base0 + j
            v01 = base0 + j2
            v10 = base1 + j
            v11 = base1 + j2

            faces.append([v00, v10, v11])
            faces.append([v00, v11, v01])

    # Convert from voxel space uniformly back to real millimeters
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing[0],
        faces=numpy.asarray(faces),
        process=False
    )

    return mesh

In [ ]:
import numpy as numpy
import trimesh
from scipy.spatial import cKDTree, Delaunay
from scipy.interpolate import UnivariateSpline

# =========================================================================
# PRODUCTION EXECUTION PIPELINE LOOP
# =========================================================================

TARGET_N_THETA = 40  # Angular configuration
ALPHA_VAL = 0.35
SLAB_THICKNESS=10

for muscle_id in range(len(muslce_ids)):

    print(f"Processing muscle {muslce_ids[muscle_id]} using Alpha Tube logic...")
    
    # Load original mesh asset
    init_mesh = trimesh.load(export_name + "_surf_id_" + str(muslce_ids[muscle_id]) + ".stl")

    # Step A: Extract pure, angle-aligned radius arrays along the original backbone coordinates
    raw_radii = extract_alpha_path_profiles_as_radii(
        centerlines[muscle_id], 
        init_mesh, 
        pixel_spacing, 
        slab_thickness=SLAB_THICKNESS,
        n_theta=TARGET_N_THETA, 
        alpha=ALPHA_VAL
    )

    # Step B: Determine trimmed spatial boundaries via intersection check
    intersection_idx = find_eye_intersection_index(
        extended_centerlines_1[muscle_id],
        mask_eye
    )
    
    # Isolate the final target trimmed backbone centerline structure
    centerline_trimmed = extended_centerlines_1[muscle_id][:intersection_idx + 1]

    # Step C: Smooth the radial array profiles across the length of the trimmed/extended line
    smoothed_radii = smooth_alpha_radii(
        raw_radii,
        n_output=len(centerline_trimmed),
        smoothing=100
    )

    # Handle fallback case if a muscle section completely failed data parsing
    if smoothed_radii is None or len(smoothed_radii) == 0:
        print(f"Skipping muscle {muslce_ids[muscle_id]} due to empty data profile.")
        continue

    # Step D: Call the updated, structurally aligned tube mesher function
    mesh = generate_alpha_tube_mesh(
        centerline_trimmed,
        smoothed_radii,
        pixel_spacing,
        n_theta=TARGET_N_THETA,
        taper_start_len=12,
        taper_end_len=12
    )

    # Step E: Export clean 3D STL file
    output_filename = export_name + "_ALPHA_TUBE_id_" + str(muslce_ids[muscle_id]) + ".stl"
    mesh.export(output_filename)
    print(f"Successfully generated and saved: {output_filename}")

Processing muscle 2 using Alpha Tube logic...
Successfully generated and saved: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_2.stl
Processing muscle 3 using Alpha Tube logic...
Successfully generated and saved: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_3.stl
Processing muscle 4 using Alpha Tube logic...
Successfully generated and saved: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_4.stl
Processing muscle 5 using Alpha Tube logic...
Successfully generated and saved: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_5.stl


In [187]:
h=1.0

out_folder = "re-meshed_h_"+str(h).replace(".","-")+"/"

if not os.path.isdir(base + folder + out_folder):
    os.mkdir(base + folder + out_folder)


for muscle_id in range(len(muslce_ids)):

    tetra_mesh_from_stl(export_name + "_ALPHA_TUBE_id_" + str(muslce_ids[muscle_id]) + ".stl", base + folder + out_folder + "id_"+str(muslce_ids[muscle_id])+"_" + "_h_"+str(h).replace(".","-"), h)

Info    : Reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_2.stl'...
Info    : Mesh is in binary format
Info    : 10720 facets in solid 0 
Info    : Done reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Alpha-spline_meshes/Segmentation_1._ALPHA_TUBE_id_2.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 10640 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 5216 triangles split in 2 parts because parametrized triangles are too small (9.73124e-09)
Info    :  - Level 2 partition with 2556 triangles split in 2 parts because parametrized triangles are too small (8.91799e-09)
Info    :  - Level 1 partition with 5424 triangles split in 2 parts because parametrized triangles are too small (1.11383e-1

Info    : Done meshing 1D (Wall 0.00159412s, CPU 0.001676s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 2 (Discrete surface, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 3 (Discrete surface, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 4 (Discrete surface, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 5 (Discrete surface, Frontal-Delaunay)
Info    : [ 60%] Triangle not found for first derivative at uv=(0.199305,-0.000812902) on discrete surface 5
Info    : [ 60%] Triangle not found for first derivative at uv=(0.199305,-0.000812902) on discrete surface 5
Info    : [ 70%] Meshing surface 6 (Discrete surface, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 7 (Discrete surface, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.64698s, CPU 0.645901s)
Info    : Meshing 3D...
Info    : Meshing volume 1 (Frontal)
Info    : Region 1 Face 2, 0 intersect
Info    : Region 1 Face 3, 0 intersect
Info    : Region 1 Face 4, 0 intersect
Info    : Region 1 Face 5, 0 in

Info    :  - Level 1 partition with 5066 triangles split in 2 parts because parametrized triangles are too small (9.51494e-09)
Info    :  - Level 2 partition with 2483 triangles split in 2 parts because parametrized triangles are too small (7.52207e-09)
Info    :  - Level 1 partition with 5014 triangles split in 2 parts because parametrized triangles are too small (1.94203e-12)
Info    :  - Level 2 partition with 2556 triangles split in 2 parts because parametrized triangles are too small (9.22016e-09)
Info    :  - Level 2 partition with 2458 triangles split in 2 parts because parametrized triangles are too small (6.07718e-09)
Info    : Found 7 model surfaces
Info    : Found 9 model curves
Info    : Done classifying surfaces (Wall 0.155122s, CPU 0.151587s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 2.17087e-05s, CPU 2.6e-05s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete 

Info    : [ 30%] Meshing surface 3 (Discrete surface, Frontal-Delaunay)
Info    : [ 50%] Meshing surface 4 (Discrete surface, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 5 (Discrete surface, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 6 (Discrete surface, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.451734s, CPU 0.451257s)
Info    : Meshing 3D...
Info    : Meshing volume 1 (Frontal)
Info    : Region 1 Face 2, 0 intersect
Info    : Region 1 Face 3, 0 intersect
Info    : Region 1 Face 4, 0 intersect
Info    : Region 1 Face 5, 0 intersect
Info    : Region 1 Face 6, 0 intersect
Info    : CalcLocalH: 1123 Points 0 Elements 2242 Surface Elements 
Info    : Check subdomain 1 / 1 
Info    : 2242 open elements 
Info    : Meshing subdomain 1 of 1 
Info    : 2242 open elements 
Info    : Use internal rules 
Info    : 2242 open elements 
Info    : Delaunay meshing 
Info    : number of points: 1123 
Info    : blockfill local h 
Info    : number of points: 1806 
Info    : Points

Info    : [100%] Discrete surface 11 is planar, simplifying parametrization                                       
Info    : Done creating geometry of discrete surfaces (Wall 0.0348495s, CPU 0.034186s)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 17 (Discrete curve)
Info    : [ 10%] Meshing curve 18 (Discrete curve)
Info    : [ 20%] Meshing curve 19 (Discrete curve)
Info    : [ 20%] Meshing curve 20 (Discrete curve)
Info    : [ 30%] Meshing curve 21 (Discrete curve)
Info    : [ 30%] Meshing curve 22 (Discrete curve)
Info    : [ 40%] Meshing curve 23 (Discrete curve)
Info    : [ 40%] Meshing curve 24 (Discrete curve)
Info    : [ 50%] Meshing curve 25 (Discrete curve)
Info    : [ 60%] Meshing curve 26 (Discrete curve)
Info    : [ 60%] Meshing curve 27 (Discrete curve)
Info    : [ 70%] Meshing curve 28 (Discrete curve)
Info    : [ 70%] Meshing curve 29 (Discrete curve)
Info    : [ 80%] Meshing curve 30 (Discrete curve)
Info    : [ 80%] Meshing curve 31 (Discrete curve)
Info    :

In [188]:
# def smooth_alpha_radii_with_mask(radii_all, n_output, smoothing=0.2):
#     """
#     Fits splines for extrapolation but also outputs a boolean mask 
#     indicating exactly where real raw data profiles existed.
#     """
#     n_input = len(radii_all)
    
#     valid_slice = next((r for r in radii_all if r is not None), None)
#     if valid_slice is None:
#         print("Warning: All slices in this muscle returned None!")
#         return None, None
#     n_theta = len(valid_slice)
    
#     # Track exactly where real data slices are (True = real data, False = empty/None)
#     data_mask = numpy.array([r is not None for r in radii_all])
    
#     radii_matrix = numpy.array([r if r is not None else numpy.full(n_theta, numpy.nan) for r in radii_all])
    
#     # Fill internal empty slices cleanly
#     for j in range(n_theta):
#         valid_mask = ~numpy.isnan(radii_matrix[:, j])
#         if numpy.any(valid_mask):
#             radii_matrix[:, j] = numpy.interp(
#                 numpy.arange(n_input), 
#                 numpy.where(valid_mask)[0], 
#                 radii_matrix[valid_mask, j]
#             )
#         else:
#             radii_matrix[:, j] = 0.1

#     s_in = numpy.linspace(0, 1, n_input)
#     s_out = numpy.linspace(0, 1, n_output)

#     smoothed_radii = numpy.zeros((n_output, n_theta))
    
#     # Fit splines (used for the extrapolated zones)
#     for j in range(n_theta):
#         spline = UnivariateSpline(s_in, radii_matrix[:, j], k=3, s=smoothing)
#         smoothed_radii[:, j] = spline(s_out)
        
#     return smoothed_radii, data_mask


# def generate_hybrid_alpha_mesh(
#         centerline,
#         raw_radii,         # Pass the original list of raw extracted profiles
#         radii_smooth,      # Pass the calculated spline matrix
#         data_mask,         # Pass the boolean data mask
#         pixel_spacing,
#         n_theta=24,
#         taper_start_len=12,
#         taper_end_len=12):
#     """
#     Constructs a 3D tube mesh using exact raw alpha radii where known,
#     and falls back to spline extrapolation for extended centerline zones.
#     """
#     n_sections = len(centerline)
#     n_input_raw = len(raw_radii)
#     vertices = []

#     theta = numpy.linspace(0, 2 * numpy.pi, n_theta, endpoint=False)
#     start_point = centerline[0]
#     end_point = centerline[-1]

#     # Map output slices back onto the relative index spacing of the raw input data
#     # to figure out exactly which input slice matches our current output slice
#     output_to_input_mapping = numpy.linspace(0, n_input_raw - 1, n_sections)

#     for i in range(n_sections):
#         direction = (
#             centerline[min(i+1, n_sections-1)]
#             - centerline[max(i-1, 0)]
#         )
#         direction /= (numpy.linalg.norm(direction) + 1e-8)
#         v1, v2 = get_local_frame(direction)

#         # Map current slice position back to the input index
#         mapped_idx = int(round(output_to_input_mapping[i]))
        
#         # -----------------------------------------------------------------
#         # HYBRID RADIUS SELECTOR: Raw vs. Spline Extrapolated
#         # -----------------------------------------------------------------
#         if mapped_idx < n_input_raw and data_mask[mapped_idx]:
#             # Use your absolute exact raw data profile!
#             r = raw_radii[mapped_idx].copy()
#         else:
#             # Fall back to smooth spline extrapolation for extended areas
#             r = radii_smooth[i].copy()

#         r = numpy.maximum(r, 0.05)
#         P = centerline[i]

#         # Proximal End Taper Math
#         if i < taper_start_len:
#             t = i / max(taper_start_len - 1, 1)
#             scale = t**2
#             P = (1 - t) * start_point + t * P
#             r *= scale

#         # Distal End Taper Math
#         if i >= n_sections - taper_end_len:
#             t = (i - (n_sections - taper_end_len)) / max(taper_end_len - 1, 1)
#             scale = (1 - t)**2
#             P = (1 - t) * P + t * end_point
#             r *= scale

#         # Standard Ring projection
#         ring = (
#             P[None, :]
#             + r[:, None] * numpy.cos(theta)[:, None] * v1[None, :]
#             + r[:, None] * numpy.sin(theta)[:, None] * v2[None, :]
#         )
#         vertices.append(ring)

#     # Tube Face Topology (Unchanged from your baseline code)
#     vertices = numpy.vstack(vertices)
#     faces = []
#     for i in range(n_sections - 1):
#         base0, base1 = i * n_theta, (i + 1) * n_theta
#         for j in range(n_theta):
#             j2 = (j + 1) % n_theta
#             faces.append([base0 + j, base1 + j, base1 + j2])
#             faces.append([base0 + j, base1 + j2, base0 + j2])

#     mesh = trimesh.Trimesh(vertices=vertices * pixel_spacing[0], faces=numpy.asarray(faces), process=False)
#     return mesh


# TARGET_N_THETA = 24  
# ALPHA_VAL = 0.15

# for muscle_id in range(len(muslce_ids)):

#     print(f"Processing hybrid model for muscle {muslce_ids[muscle_id]}...")
#     init_mesh = trimesh.load(export_name + "_surf_id_" + str(muslce_ids[muscle_id]) + ".stl")

#     # Step A: Gather original radius layout profiles
#     raw_radii = extract_alpha_path_profiles_as_radii(
#         centerlines[muscle_id], init_mesh, pixel_spacing, n_theta=TARGET_N_THETA, alpha=ALPHA_VAL
#     )

#     intersection_idx = find_eye_intersection_index(extended_centerlines_1[muscle_id], mask_eye)
#     centerline_trimmed = extended_centerlines_1[muscle_id][:intersection_idx + 1]

#     # Step B: Compute the spline shapes and grab the raw validity mask matrix
#     smoothed_radii, data_mask = smooth_alpha_radii_with_mask(
#         raw_radii,
#         n_output=len(centerline_trimmed),
#         smoothing=0.2
#     )

#     if smoothed_radii is None:
#         continue

#     # Step C: Render using your brand new hybrid meshing function
#     mesh = generate_hybrid_alpha_mesh(
#         centerline_trimmed,
#         raw_radii=raw_radii,         # Feeds raw data row arrays
#         radii_smooth=smoothed_radii,   # Feeds spline fallbacks
#         data_mask=data_mask,           # Feeds the routing tracker
#         pixel_spacing=pixel_spacing,
#         n_theta=TARGET_N_THETA,
#         taper_start_len=12,
#         taper_end_len=12
#     )

#     output_filename = export_name + "_HYBRID_ALPHA_TUBE_id_" + str(muslce_ids[muscle_id]) + ".stl"
#     mesh.export(output_filename)
#     print(f"Successfully exported hybrid structure to: {output_filename}")